# TP3 - Optimización de Modelos de Machine Learning

**Objetivo:** predecir si una reserva de infraestructura GPU será cancelada antes de su ejecución (`canceled_job`).

**Métrica de la competencia:** Balanced Accuracy.

## 1. Exploración inicial del dataset

En esta primera etapa recorremos el dataset para entender su estructura: cantidad de registros, variables disponibles, tipo de cada variable, valores faltantes y categorías.

En esta etapa **no** analizamos la relación de las variables con el target. Ese análisis se hará después de separar train y test, para evitar data leakage.

In [1]:
# IMPORTS

import pandas as pd
# pandas sirve para trabajar con datos en forma de tablas (DataFrames).

import numpy as np
import matplotlib.pyplot as plt

# Para que pandas muestre todas las columnas al imprimir un DataFrame
pd.set_option("display.max_columns", None)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# CARGA DEL DATASET

# El CSV está en la carpeta data/, un nivel arriba de notebooks/
df = pd.read_csv("../data/gpu_resource_reservations.csv")

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

### Resumen de las variables

Para cada columna calculamos:
- **tipo_pandas:** cómo la interpreta pandas (int, float, texto).
- **valores_unicos:** cuántos valores distintos toma. Ayuda a distinguir variables continuas, discretas y categóricas.
- **nulos / porc_nulos:** cantidad y porcentaje de valores faltantes.
- **ejemplo:** el valor de la primera fila.

In [ ]:
# TABLA RESUMEN DE TODAS LAS VARIABLES

resumen = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "nulos": df.isna().sum(),
    "porc_nulos": (df.isna().mean() * 100).round(2),
    "ejemplo": df.iloc[0],
})

resumen

In [ ]:
# CLASIFICACIÓN CONCEPTUAL DE CADA VARIABLE

tipos_variables = {
    "canceled_job": "Target (binaria)",
    "hours_before_execution": "Numérica discreta",
    "execution_year": "Temporal",
    "execution_month": "Temporal",
    "execution_week": "Temporal",
    "execution_day": "Temporal",
    "peak_gpu_hours": "Numérica discreta",
    "offpeak_gpu_hours": "Numérica discreta",
    "num_primary_processes": "Numérica discreta",
    "num_auxiliary_processes": "Numérica discreta",
    "num_background_processes": "Numérica discreta",
    "service_tier": "Categórica nominal",
    "client_region": "Categórica nominal (alta cardinalidad)",
    "client_type": "Categórica nominal",
    "request_channel": "Categórica nominal",
    "returning_client": "Binaria",
    "previous_job_cancellations": "Numérica discreta",
    "successful_previous_jobs": "Numérica discreta",
    "reserved_gpu_type": "Categórica nominal",
    "assigned_gpu_type": "Categórica nominal",
    "resource_modifications": "Numérica discreta",
    "reservation_commitment": "Categórica nominal",
    "orchestration_agent_id": "Identificador (categórica, alta cardinalidad)",
    "organization_id": "Identificador (categórica, alta cardinalidad)",
    "queue_wait_time": "Numérica discreta",
    "workload_profile": "Categórica nominal",
    "estimated_cost_per_hour": "Numérica continua",
    "reserved_storage_units": "Numérica discreta",
    "special_resource_requests": "Numérica discreta",
    "execution_status": "LEAKAGE (resultado de la reserva)",
    "execution_status_date": "LEAKAGE (fecha del resultado)",
    "total_processes": "Numérica discreta (derivada)",
    "total_gpu_hours": "Numérica discreta (derivada)",
    "high_value_client": "Binaria (derivada)",
}

resumen["tipo_variable"] = resumen.index.map(tipos_variables)

resumen[["tipo_variable", "tipo_pandas", "valores_unicos", "porc_nulos"]].sort_values("tipo_variable")

In [ ]:
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO

conteo = df["canceled_job"].value_counts().sort_index()
porcentaje = df["canceled_job"].value_counts(normalize=True).sort_index() * 100

print(pd.DataFrame({"cantidad": conteo, "porcentaje": porcentaje.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No cancelada (0)", "Cancelada (1)"], conteo.values, color=["#4C72B0", "#DD8452"])

# Escribimos el valor arriba de cada barra
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v} ({porcentaje.values[i]:.1f}%)", ha="center", va="bottom")

ax.set_title("Distribución de la variable objetivo")
ax.set_ylabel("Cantidad de reservas")
plt.tight_layout()
plt.show()

In [ ]:
# CATEGORÍAS DE CADA VARIABLE DE TEXTO

# Mostramos las 10 categorías más frecuentes de cada variable categórica.
# dropna=False hace que también se cuenten los valores faltantes.

cols_categoricas = df.select_dtypes(include=["object", "string"]).columns

for col in cols_categoricas:
    print(f"\n=== {col} ({df[col].nunique()} categorías) ===")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
# VERIFICACIÓN DE LEAKAGE: execution_status vs canceled_job

pd.crosstab(df["execution_status"], df["canceled_job"])

### Observaciones de la exploración inicial

- El dataset tiene **47.842 reservas y 34 columnas** (33 sin contar el target).
- **Balance de clases:** 58,3% no canceladas y 41,7% canceladas. Hay un desbalance moderado, menor que en el TP2.
- **Valores faltantes:**
  - `organization_id` tiene un 95% de nulos. Probablemente un nulo significa "la reserva no pertenece a ninguna organización", no un error.
  - `orchestration_agent_id` tiene un 10,5% de nulos (reservas sin agente).
  - `client_region` tiene 20 nulos y `num_auxiliary_processes` tiene 1.
- **Data leakage:**
  - `execution_status` coincide exactamente con el target: "canceled" y "no_show" siempre son `canceled_job = 1`, y "completed" siempre es 0.
  - `execution_status_date` es la fecha en que se canceló o completó la reserva, información posterior al momento de la predicción.
  - **Ambas se eliminarán.**
- **Variables derivadas:** `total_processes`, `total_gpu_hours` y `high_value_client` no aparecen en la consigna y parecen construidas a partir de otras. Vamos a revisar si son redundantes.
- **Categorías poco frecuentes:**
  - Hay valores con muy pocos casos, como `GPU_T12` (6), `flexible_commitment` (12) y `enterprise` en `service_tier` (9).
  - `client_region` tiene 149 países.
  - Será necesario agruparlas.
- **`reserved_gpu_type` vs. `assigned_gpu_type`:** la GPU asignada se define cerca de la ejecución. Hay que analizar si introduce leakage.